# SegmentTable の基本

このチュートリアルでは、**GWexpy** におけるセグメント（時間区間）ベースの解析コンテナである `SegmentTable` を紹介します。通常の `EventTable` とは異なり、`SegmentTable` は区間のメタデータだけでなく、対応するペイロード（TimeSeries や ASD など）を遅延ロード（Lazy loading）形式で保持することに特化しています。

In [ ]:
import numpy as np
from gwpy.segments import Segment
from gwexpy.table import SegmentTable

# 1. セグメントの作成
segs = [Segment(0, 4), Segment(4, 8), Segment(8, 12)]
st = SegmentTable.from_segments(segs, label=["A", "B", "C"])
st

## SegmentCell による遅延ロード

必要になるまでデータをロードしない「ペイロード列」を追加できます。これは巨大なデータのバッチ処理に非常に有効です。

In [ ]:
def my_loader():
    # ロードをシミュレート
    print("Loading series...")
    from gwpy.timeseries import TimeSeries
    return TimeSeries(np.random.randn(128), sample_rate=32)

# loader（コールバック関数のリスト）を指定してペイロード列を追加
st.add_series_column("raw", loader=[my_loader]*len(st), kind="timeseries")
st

## 行単位の処理

`SegmentTable` は `apply()` メソッドを提供し、各行を処理して新しい列として統合できます。

In [ ]:
def process_row(row):
    span = row["span"]
    return {"duration": float(span[1] - span[0]), "valid": True}

st2 = st.apply(process_row)
st2.display()

## 明示的ロードとデータ変換

`fetch()` や `materialize()` を使ってデータを明示的にロードできます。`to_pandas()` を使うと、通常の pandas DataFrame として扱えます。

In [ ]:
st2.fetch()
df = st2.to_pandas()
df.head()